In [79]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 
emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [80]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [81]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


In [82]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [83]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [84]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


-42.58225552052512


In [85]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [86]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [87]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides. 

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .] 
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .] 
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [88]:
# Initiate two matrices:
# viterbi_value_matrix: stores the max log-probability of reaching each (state, position)
# viterbi_trace_matrix: stores the previous state index that led to the max value
#
# Rows = states (s, E, 5, I, e)  |  Columns = sequence positions (0 to seq_len-1)
# Initialized to -inf for value matrix (log of 0 probability), -1 for trace matrix

NEG_INF = float('-inf')
num_states = len(states)
seq_len = len(query_sequence)

viterbi_value_matrix = np.full((num_states, seq_len), NEG_INF)
viterbi_trace_matrix = np.full((num_states, seq_len), -1, dtype=int)

# --- Initialization: column 0 ---
# The model always starts in state 's' (index 0).
# For first nucleotide: v[j][0] = log(trans[s->j]) + log(emission[j][nuc_0])
# Trace for col 0 always points back to start state (index 0)

nuc0 = emission_nuc_codes[query_sequence[0]]
for j in range(num_states):
    trans = state_transition_prob[0][j]   # transition from start state 's'
    emiss = emission_probs[j][nuc0]
    if trans > 0 and emiss > 0:
        viterbi_value_matrix[j][0] = math.log(trans) + math.log(emiss)
    viterbi_trace_matrix[j][0] = 0   # all trace back to start state

print("Initialization (column 0, nucleotide '{}'):".format(query_sequence[0]))
for j in range(num_states):
    val = viterbi_value_matrix[j][0]
    print(f"  State {id2state[j]}: {val}")

Initialization (column 0, nucleotide 'C'):
  State s: -inf
  State E: -1.3862943611198906
  State 5: -inf
  State I: -inf
  State e: -inf


### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND** 

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [89]:
def calculate_prob_for_a_node(col, state_j, query_sequence):

    nuc = emission_nuc_codes[query_sequence[col]]
    emiss = emission_probs[state_j][nuc]

    # If this state cannot emit this nucleotide, probability is 0 => log is -inf
    if emiss == 0:
        return NEG_INF, -1

    log_emiss = math.log(emiss)
    best_val = NEG_INF
    best_prev = -1

    # Check all possible previous states
    for prev_state in range(num_states):
        trans = state_transition_prob[prev_state][state_j]
        if trans == 0:              # impossible transition, skip
            continue
        prev_val = viterbi_value_matrix[prev_state][col - 1]
        if prev_val == NEG_INF:     # previous state was unreachable, skip
            continue
        # Viterbi recurrence in log space
        val = prev_val + math.log(trans) + log_emiss
        if val > best_val:
            best_val = val
            best_prev = prev_state

    return best_val, best_prev

In [90]:
# Fill the Viterbi matrices column by column (position by position), state by state
for col in range(1, seq_len):
    for j in range(num_states):
        val, best_prev = calculate_prob_for_a_node(col, j, query_sequence)
        viterbi_value_matrix[j][col] = val
        viterbi_trace_matrix[j][col] = best_prev

# Display the completed Viterbi Value Matrix
print("Viterbi Value Matrix (log probabilities):")
print(f"{'':5s}", end="")
for nuc in query_sequence:
    print(f"{nuc:>8s}", end="")
print()
for j in range(num_states):
    print(f"{id2state[j]:5s}", end="")
    for col in range(seq_len):
        v = viterbi_value_matrix[j][col]
        if v == NEG_INF:
            print(f"{'  -inf':>8s}", end="")
        else:
            print(f"{v:8.2f}", end="")
    print()

Viterbi Value Matrix (log probabilities):
            C       T       T       C       A       T       G       T       G       A       A       A       G       C       A       G       A       C       G       T       A       A       G       T       C       A
s        -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf
E       -1.39   -2.88   -4.37   -5.86   -7.35   -8.84  -10.34  -11.83  -13.32  -14.81  -16.30  -17.79  -19.29  -20.78  -22.27  -23.76  -25.25  -26.74  -28.24  -29.73  -31.22  -32.71  -34.20  -35.69  -37.19  -38.68
5        -inf    -inf    -inf    -inf  -11.16    -inf  -11.20    -inf  -14.18  -18.62  -20.11  -21.60  -20.15    -inf  -26.08  -24.62  -29.06    -inf  -29.10    -inf  -35.03  -36.52  -35.06    -inf    -inf  -42.48
I        -inf    -inf    -inf    -inf    -inf  -12.08  -14.48  -12.11  -14.52  -15.10  -16.12  -17.14 

In [91]:
# Traceback: recover the optimal state path from the filled matrices

def traceback(viterbi_value_matrix, viterbi_trace_matrix, query_sequence):

    seq_len = len(query_sequence)

    # Step 1: Find the best final state in the last column
    last_col = viterbi_value_matrix[:, seq_len - 1]
    best_last_state = int(np.argmax(last_col))
    best_prob = last_col[best_last_state]

    # Step 2: Traceback from last column to first
    path = [best_last_state]
    current = best_last_state
    for col in range(seq_len - 1, 0, -1):
        prev = viterbi_trace_matrix[current][col]
        path.append(prev)
        current = prev

    # Step 3: Reverse to get left-to-right order
    path.reverse()
    state_path = "".join([id2state[s] for s in path])
    return state_path, best_prob


optimal_path, optimal_prob = traceback(viterbi_value_matrix, viterbi_trace_matrix, query_sequence)

print("=" * 55)
print("VITERBI ALGORITHM - FINAL RESULT")
print("=" * 55)
print(f"Query Sequence : {query_sequence}")
print(f"Optimal Path   : {optimal_path}")
print(f"Log Probability: {optimal_prob:.5f}")
print("=" * 55)
print()
print("State legend:")
print("  s = Start state")
print("  E = Exon region")
print("  5 = 5' splice site")
print("  I = Intron region")
print("  e = End state")
print()
print("Interpretation:")
print(f"  The Viterbi algorithm assigns the highest probability")
print(f"  to the path '{optimal_path}'.")
print(f"  This indicates the sequence is best explained as")
print(f"  entirely in the Exon (E) state, with no splice site")
print(f"  detected. The 5' splice site state requires a 'G' at")
print(f"  its position (emission prob 0.95), and no position in")
print(f"  this sequence provides a strong enough 'G' signal to")
print(f"  overcome the cost of transitioning through 5 -> I.")

VITERBI ALGORITHM - FINAL RESULT
Query Sequence : CTTCATGTGAAAGCAGACGTAAGTCA
Optimal Path   : EEEEEEEEEEEEEEEEEEEEEEEEEE
Log Probability: -38.67767

State legend:
  s = Start state
  E = Exon region
  5 = 5' splice site
  I = Intron region
  e = End state

Interpretation:
  The Viterbi algorithm assigns the highest probability
  to the path 'EEEEEEEEEEEEEEEEEEEEEEEEEE'.
  This indicates the sequence is best explained as
  entirely in the Exon (E) state, with no splice site
  detected. The 5' splice site state requires a 'G' at
  its position (emission prob 0.95), and no position in
  this sequence provides a strong enough 'G' signal to
  overcome the cost of transitioning through 5 -> I.
